# CC3104 — Reinforcement Learning
## Laboratorio 13 — Task 1

## Task 1.1

### a. DQN con acciones continuas
DQN actúa como $\pi(s)=\arg\max_{a\in\mathcal A}Q_\theta(s,a)$. Si $\mathcal A$ es continuo, hay infinitas acciones y ese $\arg\max$ no se puede enumerar. Optimizar $Q_\theta(s,a)$ respecto a $a$ en cada decisión es costoso y no convexo; discretizar el ángulo vuelve el costo exponencial al aumentar precisión/dimensiones y puede dar ajustes muy bruscos. Por eso conviene un actor continuo que produzca directamente el ángulo.

### b. TD3 vs. SAC
TD3 da una política determinista, algo atractivo para una planta: el control es repetible y más fácil de auditar. Sus *twin critics*, *delayed updates* y suavizado del target reducen sobreestimación, pero explora con ruido externo y puede cubrir peor vientos nuevos. SAC usa una política estocástica y entropía: durante simulación suele explorar y reutilizar datos mejor, y normalmente queda más robusto a perturbaciones. Esa aleatoriedad no es ideal para mandar acciones directas a hardware. Recomendaría **SAC** entrenado en simulación, desplegando la acción media con límites/safety layer; así se conserva robustez sin meter ruido innecesario en producción.

### c. ¿Sirven los datos para CQL?
Sí, como punto de partida, pero no son garantía. La política de comportamiento $\beta$ es una mezcla de operadores: probablemente segura pero no cercana a óptima. Cubrirá estados de operación comunes, aunque en un espacio continuo los ángulos humanos serán estrechos y habrá pocos ejemplos de ráfagas, fallas o maniobras límite. De las dos condiciones usuales de Offline RL, datos de buena calidad y cobertura de soporte, la más difícil es la **cobertura**: no se puede evaluar bien una acción que $\beta$ casi nunca tomó. CQL ayuda justamente a no confiar tanto en esas zonas.

### d. Riesgo de Q-Learning sin conservatividad
El mayor riesgo es el **error de extrapolación**: el Critic asigna $Q$ alto a ángulos fuera de la distribución histórica porque nunca recibió evidencia que los castigue. El actor puede perseguir ese falso máximo y ordenar un pitch extremo; operativamente eso puede elevar cargas, provocar *stall* o disparar una parada de protección, aunque el valor estimado parezca bueno.

## Task 1.2

### a. Objetivo de SAC
$$
J(\pi)=\mathbb E_{\tau\sim\pi}\left[\sum_{t=0}^{\infty}\gamma^t\left(r_t+\alpha\,\mathcal H(\pi(\cdot\mid s_t))\right)\right].
$$
Aquí $\alpha$ es la temperatura: pondera energía/seguridad contra exploración. $\mathcal H(\pi(\cdot\mid s_t))$ mide cuán dispersos son los ángulos posibles de pitch en el estado de viento $s_t$. Se ajusta $\alpha$ automáticamente para alcanzar una entropía objetivo, por ejemplo minimizando $\mathbb E[-\alpha(\log\pi(a_t\mid s_t)+\mathcal H_{target})]$; si la política se vuelve demasiado determinista, $\alpha$ sube.

### b. Target del Critic
TD3 usa
$$y_{TD3}=r+\gamma\min_{i=1,2}Q_{\bar\theta_i}(s',\pi_{\bar\phi}(s')+\epsilon),\quad \epsilon\text{ recortado}.$$
SAC usa
$$y_{SAC}=r+\gamma\,\mathbb E_{a'\sim\pi}[\min_iQ_{\bar\theta_i}(s',a')-\alpha\log\pi(a'\mid s')].$$
Ambos usan el mínimo de dos Critics. Para valor extrínseco, TD3 suele ser más conservador: toma el mínimo y además suaviza acciones target; SAC agrega el bono de entropía al valor *soft*. Esa cautela es buena si evita sobrevalorar ángulos inseguros, pero demasiada puede dejar energía disponible sin capturar.

### c. Objetivo de CQL
$$
\mathcal L_{CQL}=\underbrace{\mathbb E_{(s,a,r,s')\sim D}[(Q_\theta(s,a)-y)^2]}_{\text{TD estándar}}+\alpha_{CQL}\left(\underbrace{\mathbb E_{s\sim D}[\log\sum_a e^{Q_\theta(s,a)}]}_{\text{sube Q de acciones candidatas}}-\underbrace{\mathbb E_{(s,a)\sim D}[Q_\theta(s,a)]}_{\text{mantiene Q de acciones vistas}}\right).
$$
El primer término extra penaliza valores altos para acciones candidatas, incluidas las fuera de datos; el segundo evita bajar también las acciones que sí observamos. Juntos dejan $Q$ pesimista fuera del soporte de $D$. Por la heterogeneidad de operadores usaría un $\alpha_{CQL}$ moderado, iniciando en **1.0** y afinándolo con validación offline (por ejemplo $0.5$–$5$): hay más riesgo de acciones poco cubiertas, pero un valor enorme castigaría conductas buenas de los operadores expertos.